# Task 4: Statistical Modeling & Risk-Based Pricing

**Objective:** build and evaluate predictive models that support dynamic, risk-based pricing for ACIS.

**Modeling goals**
- Predict claim severity for policies with a claim using `TotalClaims` as the target.
- Build a claim-probability model for the pricing framework.
- Combine claim probability and predicted severity into a risk-based premium recommendation.

**Evaluation focus**
- Regression: RMSE and R².
- Classification: accuracy, precision, recall, and F1.
- Interpretability: SHAP-based feature importance for the best severity model.


## Setup and Data Loading

Load the prepared insurance dataset and the reusable modeling helpers from `src`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_loader import load_and_prepare
from modeling import (
    explain_top_features_business_terms,
    run_modeling_workflow,
)


df = load_and_prepare(str(ROOT / 'data' / 'insurance_data.csv'))
print(f'Dataset loaded: {df.shape}')
df.head()

## Model Training

The next cell trains and compares the severity models, then fits the claim-probability models used in the pricing framework.

In [ ]:
workflow = run_modeling_workflow(
    df,
    test_size=0.3,
    shap_output_path=ROOT / 'reports' / 'task4_shap_summary.png',
)

severity_results = workflow['severity_comparison']
frequency_results = workflow['frequency_comparison']

print('Severity model comparison:')
display(severity_results)
print('\nClaim-probability model comparison:')
display(frequency_results)

severity_results.to_csv(ROOT / 'reports' / 'task4_severity_model_comparison.csv', index=False)
frequency_results.to_csv(ROOT / 'reports' / 'task4_frequency_model_comparison.csv', index=False)

## Feature Importance & Interpretability

The best severity model is explained with SHAP to identify the strongest pricing drivers.

In [ ]:
feature_importance = workflow['feature_importance']
print(f"Best severity model: {workflow['best_severity_model_name']}")
display(feature_importance)

print('\nBusiness interpretation of the top features:')
for statement in explain_top_features_business_terms(feature_importance.head(5)):
    print(f'- {statement}')

## Risk-Based Pricing

The final step combines claim probability and predicted severity into a recommended premium.

In [ ]:
pricing_frame = workflow['pricing_frame']
pricing_output = pricing_frame[[
    'PolicyID',
    'TotalPremium',
    'ClaimProbability',
    'PredictedSeverity',
    'ExpectedClaim',
    'RecommendedPremium',
]].copy()

display(pricing_output.head(10))
pricing_output.to_csv(ROOT / 'reports' / 'task4_pricing_recommendations.csv', index=False)
print(f"Saved pricing recommendations to {ROOT / 'reports' / 'task4_pricing_recommendations.csv'}")

## Export Artifacts

The notebook writes the model comparison tables, SHAP plot, and premium recommendations to `reports/` for submission.

In [ ]:
# Use best model (Random Forest or XGBoost typically performs best)
best_model_name = results_df.loc[results_df['R²'].idxmax(), 'Model']
best_model = results[best_model_name]['model']

print(f"\nAnalyzing feature importance for: {best_model_name}")

# SHAP analysis
if best_model_name == 'XGBoost':
    explainer = shap.TreeExplainer(best_model)
else:  # Random Forest
    explainer = shap.TreeExplainer(best_model)

shap_values = explainer.shap_values(X_test)

print("\nTop 10 Most Influential Features:")
feature_importance = pd.DataFrame({
    'Feature': X_cols,
    'Mean_SHAP': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean_SHAP', ascending=False)

print(feature_importance.head(10).to_string(index=False))

print("\n→ SHAP plots and detailed interpretation in final report.")

## Risk-Based Pricing Framework

**Dynamic Premium Formula:**
```
Premium = (Claim Frequency × Predicted Severity) + Expense Loading + Profit Margin
```

In [ ]:
print("\nPricing Framework Implementation:")
print("\n1. Claim Frequency Model: [Binary classifier to be implemented in Task 4]")
print("2. Claim Severity Model: [Best of LR, RF, XGBoost]")
print(f"3. Expected Claim = P(Claim) × Predicted Severity")
print(f"4. Proposed Premium = Expected Claim + Expense + Margin")
print(f"   → Expense Loading: ~15% of claim expected value")
print(f"   → Profit Margin: ~20% of claim expected value")
print(f"\nExample:")
print(f"  If P(Claim)=0.30 and Predicted Severity=$5,000:")
print(f"    Expected Claim = 0.30 × $5,000 = $1,500")
print(f"    Expense = $225 (15%)")
print(f"    Margin = $300 (20%)")
print(f"    Total Premium = $2,025")

## Summary & Next Steps

✓ Models trained and evaluated  
✓ Best model identified via cross-validation  
✓ SHAP analysis completed for interpretability  
✓ Pricing framework documented  

**Final Deliverables (May 26):**
- Complete modeling notebook with all three algorithms
- SHAP visualizations and business interpretation
- Final polished report combining all four tasks